# TMDB Movie Recommendation 

Movie recommendation system using the TMDB 6000 database available on kaggle: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

This script is considerably inspired by https://github.com/campusx-official/movie-recommender-system-tmdb-dataset

In [31]:
import numpy as np
import pandas as pd

In [32]:
movies = pd.read_csv('data/tmdb_5000_movies.csv')
credits = pd.read_csv('data/tmdb_5000_credits.csv')

In [33]:
data = movies.merge(credits, on='title')
data.shape

(4809, 23)

In [34]:
data.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

## Feature Selection
- ID (id)
- Title of the movie (title) (English translation of original_title)
- Genre of the movie (genres)
- Keywords associated with the movie (keywords)
- Overview of the movie (overview)
- Cast starred in the movie (cast)
- Crew that worked for the movie (crew)
- Date of movie release (release_date)

In [35]:
df = data[['id', 'title', 'genres', 'keywords', 'overview', 'cast', 'crew', 'release_date']]

# Data Cleaning

- Exclude rows with null values because they consist of less than < 1% of the data

In [36]:
df.isnull().sum()

id              0
title           0
genres          0
keywords        0
overview        3
cast            0
crew            0
release_date    1
dtype: int64

In [37]:
df.dropna(inplace=True)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\1379821321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.dropna(inplace=True)


In [38]:
df.duplicated().sum()

0

## Cleaning up the genres and keywords column

The genres and keywords column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the names from that list.

In [39]:
def reformat(row):
    import ast
    new_row = []
    for item in ast.literal_eval(row):
        new_row.append(item['name'])

    return new_row

In [40]:
df['genres'] = df['genres'].apply(reformat)
df['keywords'] =  df['keywords'].apply(reformat)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\3702656427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genres'] = df['genres'].apply(reformat)
C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\3702656427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['keywords'] =  df['keywords'].apply(reformat)


In [41]:
df.head()

,id,title,genres,keywords,overview,cast,crew,release_date
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2007-05-19
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",A cryptic message from Bond’s past sends him o...,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",2015-10-26
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",Following the death of District Attorney Harve...,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",2012-07-16
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","John Carter is a war-weary, former military ca...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",2012-03-07


## Cleaning up the cast column

The cast column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the top 3 names from that list.

In [42]:
def reformat_cast(row):
    import ast
    new_row = []
    counter = 0
    for item in ast.literal_eval(row):
        if counter != 3:
            new_row.append(item['name'])
            counter+=1
        else:
            break

    return new_row

In [43]:
df['cast'] = df['cast'].apply(reformat_cast)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\1576142058.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['cast'] = df['cast'].apply(reformat_cast)


## Cleaning up the crew column

The crew column is a string that is inplace for a list. The string will be converted to a list in Python. Then we extract the Director's names from that list.

In [44]:
def get_director(row):
    import ast
    new_row = []
    for item in ast.literal_eval(row):
        if item['job'] == 'Director':
            new_row.append(item['name'])
            break
    
    return new_row

In [45]:
df['crew'] = df['crew'].apply(get_director)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\4103153286.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['crew'] = df['crew'].apply(get_director)


In [46]:
df.head()

,id,title,genres,keywords,overview,cast,crew,release_date
0,19995,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","In the 22nd century, a paraplegic Marine is di...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron],2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","Captain Barbossa, long believed to be dead, ha...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski],2007-05-19
2,206647,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi...",A cryptic message from Bond’s past sends him o...,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes],2015-10-26
3,49026,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i...",Following the death of District Attorney Harve...,"[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan],2012-07-16
4,49529,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel...","John Carter is a war-weary, former military ca...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton],2012-03-07


## Tokenising the overview column

In [47]:
df['overview'] = df['overview'].apply(lambda x:x.split())

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\2950014412.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['overview'] = df['overview'].apply(lambda x:x.split())


## Discretising the release_date column

We want to discretise the release_date column so that we can split the dates into decades.

In [48]:
def release_decade(row):
    year = pd.to_datetime(row).year
    decade = int(np.floor(year/10) * 10)

    return decade

In [49]:
df['release_date'] = df['release_date'].apply(release_decade)

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\1524738953.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['release_date'] = df['release_date'].apply(release_decade)


## Text Preprocessing

In [50]:
df['genres'] = df['genres'].apply(lambda x: [i.replace(" ", "") for i in x ])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ", "") for i in x ])

df['cast'] = df['cast'].apply(lambda x: [i.replace(" ", "") for i in x ])
df['crew'] = df['crew'].apply(lambda x: [i.replace(" ", "") for i in x ])

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\3431474976.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['genres'] = df['genres'].apply(lambda x: [i.replace(" ", "") for i in x ])
C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\3431474976.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ", "") for i in x ])
C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\3431474976.py:4: SettingWithCopyWarning: 
A value is trying to be set on a c

In [51]:
df['tags'] = df['genres'] + df['overview'] + df['keywords'] + df['cast'] + df['crew']

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\4127071975.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['genres'] + df['overview'] + df['keywords'] + df['cast'] + df['crew']


In [52]:
df['tags'] = df['tags'].apply(lambda x: " ".join(x))
df['tags'] = df['tags'].apply(lambda x: x.lower())

C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\1778811128.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: " ".join(x))
C:\Users\eshaa\AppData\Local\Temp\ipykernel_17092\1778811128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: x.lower())


In [53]:
text_data = df[['id', 'title', 'tags']]

## Vectorisation

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    new_text = []

    for word in text.split():
        new_text.append(ps.stem(word))
    
    return " ".join(new_text)
        


tftidf = TfidfVectorizer(max_features=10000, stop_words='english')
cv = CountVectorizer(max_features=10000, stop_words='english')

In [55]:
tftidf_vectors = tftidf.fit_transform(text_data.tags).toarray()
cv_vectors = cv.fit_transform(text_data.tags).toarray()

tftidf.get_feature_names_out()

array(['000', '007', '10', ..., 'zooey', 'zooeydeschanel', 'zookeeper'],
      dtype=object)

In [56]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_tfidf = cosine_similarity(tftidf_vectors)
similarity_cv = cosine_similarity(cv_vectors)

# Recommendation

In [57]:
def recommend(title, n, text_data=text_data):
    ix = text_data[text_data['title'] == title].index[0]
    distances = similarity_tfidf[ix]
    list_of_recommendations = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1])[1:]

    counter = 1
    top_n = []
    for movie_idx in list_of_recommendations:
        if counter < n+1:
            counter+=1
            top_n.append(text_data.iloc[movie_idx[0]].title)

        else:
            break
        
    return top_n

In [58]:
import pickle
pickle.dump(text_data.to_dict(), open('data/movies.pkl', 'wb'))
pickle.dump(similarity_tfidf, open('data/similarity.pkl', 'wb'))